# Criação dos dados

In [25]:
import numpy as np
import csv

n=16000             # Quantidade de pontos
k=9000              # Quantidade de centróides
np.random.seed(2112)

def gerar_dados(N):
    faixas = [
        np.random.uniform(0, 2,   N//4),
        np.random.uniform(10,12,  N//4),
        np.random.uniform(20,22,  N//4),
        np.random.uniform(30,32,  N - 3*(N//4))
    ]
    return np.concatenate(faixas)


# Gera os pontos e os centróides iniciais
dados = gerar_dados(n)
centroides = np.random.uniform(0, 1000, k)

# Salva os dados em data.csv
with open("dados.csv", "w", newline="") as f:
    writer = csv.writer(f)
    for x in dados:
        writer.writerow([x])

# Salva os centróides em centroides_iniciais.csv
with open("centroides_iniciais.csv", "w", newline="") as f:
    writer = csv.writer(f)
    for c in centroides:
        writer.writerow([c])

print(f" {n} pontos e {k} centróides")

 16000 pontos e 9000 centróides


# Etapa 2: CUDA (GPU) - kmeans_1d_cuda.cu

In [ ]:
%%writefile kmeans_1d_cuda.cu

/* kmeans_1d_cuda.cu
   K-means 1D com aceleração CUDA.
   - Assignment: Paralelizado na GPU (1 thread por ponto).
   - Update: Sequencial na CPU (copia vetor de assign de volta).
   - SSE: Calculado na CPU somando erros individuais retornados pela GPU.
   - Medição detalhada de tempos: H2D, D2H, Kernel, Total

   Compilar: nvcc -O2 kmeans_1d_cuda.cu -o kmeans_1d_cuda
*/

#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>
#include <cuda_runtime.h>

#define THREADS_PER_BLOCK 256

#define cudaCheckError(ans) { gpuAssert((ans), __FILE__, __LINE__); }
inline void gpuAssert(cudaError_t code, const char *file, int line, bool abort=true){
   if (code != cudaSuccess){
      fprintf(stderr,"GPUassert: %s %s %d\n", cudaGetErrorString(code), file, line);
      if (abort) exit(code);
   }
}

static int count_rows(const char *path){
    FILE *f = fopen(path, "r");
    if(!f){ fprintf(stderr,"Erro ao abrir %s\n", path); exit(1); }
    int rows=0; char line[8192];
    while(fgets(line,sizeof(line),f)){
        int only_ws=1;
        for(char *p=line; *p; p++){
            if(*p!=' ' && *p!='\t' && *p!='\n' && *p!='\r'){ only_ws=0; break; }
        }
        if(!only_ws) rows++;
    }
    fclose(f);
    return rows;
}

static double *read_csv_1col(const char *path, int *n_out){
    int R = count_rows(path);
    if(R<=0){ fprintf(stderr,"Arquivo vazio: %s\n", path); exit(1); }
    double *A = (double*)malloc((size_t)R * sizeof(double));
    if(!A){ fprintf(stderr,"Sem memoria host\n"); exit(1); }
    FILE *f = fopen(path, "r");
    char line[8192]; int r=0;
    while(fgets(line,sizeof(line),f)){
        int only_ws=1;
        for(char *p=line; *p; p++){
            if(*p!=' ' && *p!='\t' && *p!='\n' && *p!='\r'){ only_ws=0; break; }
        }
        if(only_ws) continue;
        char *tok = strtok(line, ",; \t");
        if(tok) A[r++] = atof(tok);
        if(r>=R) break;
    }
    fclose(f);
    *n_out = R;
    return A;
}

static void write_assign_csv(const char *path, const int *assign, int N){
    if(!path) return;
    FILE *f = fopen(path, "w");
    if(!f) return;
    for(int i=0;i<N;i++) fprintf(f, "%d\n", assign[i]);
    fclose(f);
}

static void write_centroids_csv(const char *path, const double *C, int K){
    if(!path) return;
    FILE *f = fopen(path, "w");
    if(!f) return;
    for(int c=0;c<K;c++) fprintf(f, "%.6f\n", C[c]);
    fclose(f);
}

__global__ void assignment_kernel(const double *X, const double *C,
                                  int *assign, double *point_sse,
                                  int N, int K)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < N) {
        double myX = X[i];
        int best_k = -1;
        double min_dist_sq = 1e30; 

        for (int c = 0; c < K; c++) {
            double diff = myX - C[c];
            double dist_sq = diff * diff;
            if (dist_sq < min_dist_sq) {
                min_dist_sq = dist_sq;
                best_k = c;
            }
        }
        assign[i] = best_k;
        point_sse[i] = min_dist_sq; 
    }
}

static void update_step_cpu(const double *X, double *C, const int *assign, int N, int K){
    double *sum = (double*)calloc(K, sizeof(double));
    int *cnt = (int*)calloc(K, sizeof(int));

    for(int i=0; i<N; i++){
        int c = assign[i];
        sum[c] += X[i];
        cnt[c]++;
    }

    for(int c=0; c<K; c++){
        if(cnt[c] > 0) C[c] = sum[c] / (double)cnt[c];
        else           C[c] = X[0]; 
    }
    free(sum); free(cnt);
}

int main(int argc, char **argv){
    if(argc < 3){
        printf("Uso: %s dados.csv centroides_iniciais.csv [max_iter] [eps] [threads_per_block] [assign.csv] [centroids.csv]\n", argv[0]);
        return 1;
    }

    const char *pathX = argv[1];
    const char *pathC = argv[2];
    int max_iter = (argc>3)? atoi(argv[3]) : 50;
    double eps   = (argc>4)? atof(argv[4]) : 1e-4;
    int threads_per_block = (argc>5)? atoi(argv[5]) : THREADS_PER_BLOCK;
    const char *outAssign = (argc>6)? argv[6] : NULL;
    const char *outCentroid = (argc>7)? argv[7] : NULL;

    int N=0, K=0;
    double *h_X = read_csv_1col(pathX, &N);
    double *h_C = read_csv_1col(pathC, &K);
    int *h_assign = (int*)malloc(N * sizeof(int));
    double *h_point_sse = (double*)malloc(N * sizeof(double)); 

    double *d_X, *d_C, *d_point_sse;
    int *d_assign;

    cudaCheckError( cudaMalloc((void**)&d_X, N * sizeof(double)) );
    cudaCheckError( cudaMalloc((void**)&d_C, K * sizeof(double)) );
    cudaCheckError( cudaMalloc((void**)&d_assign, N * sizeof(int)) );
    cudaCheckError( cudaMalloc((void**)&d_point_sse, N * sizeof(double)) );

    clock_t t_h2d_init = clock();
    cudaCheckError( cudaMemcpy(d_X, h_X, N * sizeof(double), cudaMemcpyHostToDevice) );
    clock_t t_h2d_init_end = clock();
    double ms_h2d_init = 1000.0 * (double)(t_h2d_init_end - t_h2d_init) / (double)CLOCKS_PER_SEC;

    if(threads_per_block <= 0 || threads_per_block > 1024){
        fprintf(stderr, "AVISO: threads_per_block=%d inválido, usando padrão %d\n",
                threads_per_block, THREADS_PER_BLOCK);
        threads_per_block = THREADS_PER_BLOCK;
    }

    int blockSize = threads_per_block;
    int gridSize = (N + blockSize - 1) / blockSize;

    printf("K-means 1D (CUDA) - N=%d, K=%d, Grid=%d blocks, Block=%d threads\n", N, K, gridSize, blockSize);
    printf("Transferência inicial H2D: %.2f ms\n", ms_h2d_init);
    printf("threads_per_block,iteracoes,sse,h2d_ms,d2h_ms,kernel_ms,total_ms,overhead_ms\n");

    double total_h2d = ms_h2d_init;  
    double total_d2h = 0.0;
    double total_kernel = 0.0;

    clock_t t0 = clock();
    double prev_sse = 1e300;
    double sse = 0.0;
    int it = 0;

    for(it = 0; it < max_iter; it++){

        clock_t t_h2d = clock();
        cudaCheckError( cudaMemcpy(d_C, h_C, K * sizeof(double), cudaMemcpyHostToDevice) );
        clock_t t_h2d_end = clock();
        double ms_h2d = 1000.0 * (double)(t_h2d_end - t_h2d) / (double)CLOCKS_PER_SEC;
        total_h2d += ms_h2d;

        clock_t t_kernel = clock();
        assignment_kernel<<<gridSize, blockSize>>>(d_X, d_C, d_assign, d_point_sse, N, K);
        cudaCheckError( cudaGetLastError() );
        cudaCheckError( cudaDeviceSynchronize() );
        clock_t t_kernel_end = clock();
        double ms_kernel = 1000.0 * (double)(t_kernel_end - t_kernel) / (double)CLOCKS_PER_SEC;
        total_kernel += ms_kernel;

        clock_t t_d2h = clock();
        cudaCheckError( cudaMemcpy(h_assign, d_assign, N * sizeof(int), cudaMemcpyDeviceToHost) );
        cudaCheckError( cudaMemcpy(h_point_sse, d_point_sse, N * sizeof(double), cudaMemcpyDeviceToHost) );
        clock_t t_d2h_end = clock();
        double ms_d2h = 1000.0 * (double)(t_d2h_end - t_d2h) / (double)CLOCKS_PER_SEC;
        total_d2h += ms_d2h;

        sse = 0.0;
        for(int i=0; i<N; i++) sse += h_point_sse[i];

        double rel = fabs(sse - prev_sse) / (prev_sse > 0.0 ? prev_sse : 1.0);
        if(rel < eps){
            it++; 
            break;
        }
        prev_sse = sse;

        update_step_cpu(h_X, h_C, h_assign, N, K);
    }

    clock_t t1 = clock();
    double ms_total = 1000.0 * (double)(t1 - t0) / (double)CLOCKS_PER_SEC;

    printf("%d,%d,%.6f,%.2f,%.2f,%.2f,%.2f,%.2f\n",
           threads_per_block, it, sse, total_h2d, total_d2h, total_kernel, ms_total,
           ms_total - total_h2d - total_d2h - total_kernel);

    write_assign_csv(outAssign, h_assign, N);
    write_centroids_csv(outCentroid, h_C, K);

    free(h_X); free(h_C); free(h_assign); free(h_point_sse);
    cudaFree(d_X); cudaFree(d_C); cudaFree(d_assign); cudaFree(d_point_sse);

    return 0;
}

Overwriting kmeans_1d_cuda.cu


In [27]:
%%shell

nvcc -O2 -arch=sm_75 kmeans_1d_cuda.cu -o kmeans_1d_cuda

echo "=== Executando com THREADS_PER_BLOCK = 128 ==="
for i in {1..20}; do
  echo "Execução $i/20 (128 threads):"
  ./kmeans_1d_cuda dados.csv centroides_iniciais.csv 50 0.000001 128 assign_cuda.csv centroids_cuda.csv
  echo ""
done

echo "=== Executando com THREADS_PER_BLOCK = 256 ==="
for i in {1..20}; do
  echo "Execução $i/20 (256 threads):"
  ./kmeans_1d_cuda dados.csv centroides_iniciais.csv 50 0.000001 256 assign_cuda.csv centroids_cuda.csv
  echo ""
done

echo "=== Executando com THREADS_PER_BLOCK = 512 ==="
for i in {1..20}; do
  echo "Execução $i/20 (512 threads):"
  ./kmeans_1d_cuda dados.csv centroides_iniciais.csv 50 0.000001 512 assign_cuda.csv centroids_cuda.csv
  echo ""
done

=== Executando com THREADS_PER_BLOCK = 128 ===
Execução 1/20 (128 threads):
K-means 1D (CUDA) - N=16000, K=9000, Grid=125 blocks, Block=128 threads
Transferência inicial H2D: 0.06 ms
threads_per_block,iteracoes,sse,h2d_ms,d2h_ms,kernel_ms,total_ms,overhead_ms
128,50,24.665393,1.02,3.81,440.02,447.67,2.81

Execução 2/20 (128 threads):
K-means 1D (CUDA) - N=16000, K=9000, Grid=125 blocks, Block=128 threads
Transferência inicial H2D: 0.06 ms
threads_per_block,iteracoes,sse,h2d_ms,d2h_ms,kernel_ms,total_ms,overhead_ms
128,50,24.665393,0.95,3.47,260.10,267.17,2.66

Execução 3/20 (128 threads):
K-means 1D (CUDA) - N=16000, K=9000, Grid=125 blocks, Block=128 threads
Transferência inicial H2D: 0.06 ms
threads_per_block,iteracoes,sse,h2d_ms,d2h_ms,kernel_ms,total_ms,overhead_ms
128,50,24.665393,1.01,3.57,259.18,266.54,2.77

Execução 4/20 (128 threads):
K-means 1D (CUDA) - N=16000, K=9000, Grid=125 blocks, Block=128 threads
Transferência inicial H2D: 0.06 ms
threads_per_block,iteracoes,sse,h2d_m